In [ ]:
# 모! 듈! 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 그래프 기본 테마 설정
sns.set_theme(palette="Blues", style="whitegrid", font_scale=1) # 블루톤

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Pretendard' # 프리텐다드
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14 # 기본 폰트사이즈 (자세한 양식은 하기)
plt.rcParams['axes.unicode_minus'] = False

# 그래프 공통 적용 사항
# 1. 테마: 커스텀 팔레트('#5ca0de','#9dadbc')/히트맵: Blues
# 2. Font: 프리텐다드
# 3. 제목/라벨 크기: 16, 14(별도 설정이 있는 경우 12)
# 4. 제목 라벨: y = 1.03

# 읽기 전에
1. 그래프 색이 PPT에서 봤던거랑 다르고 연파랑이다=PPT에는 안 쓴 겁니다. 그냥 하다보니 겹쳐서 다른 분 거 썼구나(혹은 내용상 애매해서 안 넣었다던가...) 하시면 됩니다. 
2. 개인 깃헙에는 출력 정리 버전이 올라갈 예정입니다. 
3. 주석에 숫자가 달려있거나 파라미터가 세로로 나열된 경우(혹은 탭이 뻘건색인 경우): 제미나이가 도와줬습니다. 제미나이랑 저랑 스타일이 안 맞는 부분이 좀 있어서 차이가 있습니다. 

In [ ]:
# 일단 불러야 뭘 한다
bank_df = pd.read_csv('data/bank_preprocess.csv')

# 저장하면서 뭔가 잘못된 것 같습니다. 
# 이거 개인적으로 캐글 EDA 할 때도 이랬습니다. 
bank_df.drop('Unnamed: 0', axis=1, inplace=True)

bank_df

# 프로젝트의 목적과 가설
## 목적 
- 예금 가입 전환을 개선하기 위한 전략 도출 목적의 가설 수립 및 분석 진행
- 연령별 최종 예금 가입 전환율

## 가설
- 특정 연령대(3~40대)인 고객들의 가입률이 높을 것이다. 
- 3~40대라고 생각한 이유: 결혼자금, 노후준비, (아이가 있다면) 아이의 마래를 위한 적금, 비상금 등.. 
- 3040은 직장에서 어느정도 자리를 잡아가는 연령대이기도 하고, 금융 상품을 주도적으로 비교 및 선택할 수 있는 연령대이기도 하다. 

# 연령대별 연락 비율 

In [ ]:
# 연령대로 그룹바이
bank_df_age = bank_df.groupby('age_group')[['campaign']].count().sort_values('campaign', ascending=False)

# Pie chart로 비율 확인 
# 어떻게 해도 안이쁨.. 
def my_fmt(x):
    return '{:.1f}%'.format(x) if x > 1 else ''

plt.figure(figsize=(12, 8))
plt.pie(bank_df_age['campaign'], labels = bank_df_age.index, autopct=my_fmt, startangle=140, pctdistance = 0.85, explode=[0.05]*len(bank_df_age))
plt.title('Campaign distribution by Age', fontsize = 16, y = 1.03)

# 비율이 많은 연령대(2~50)대와 그렇지 않은 연령대의 비율 차이 확인 
main_age_group = [20, 30, 40, 50] # 2, 3, 4, 50대
bank_df['group_category'] = bank_df['age_group'].apply(lambda x: f"{int(x)}대" if x in main_age_group else '기타(Others)') 
# 저 안에 있는 람다는 익명함수, 혹은 람다식이라고 부릅니다. 저는 김람다씨라고 부르고 있죠. 

# 20~50대+기타로 묶음
# 여기도 파이차트
group_counts = bank_df['group_category'].value_counts()
plt.figure(figsize=(12, 8))
plt.pie(group_counts, labels=group_counts.index, autopct='%1.1f%%', startangle=140, explode=[0.05]*len(group_counts))
plt.title('Campaign Distribution: Main Age Groups vs Others', fontsize = 16, y = 1.03)
plt.show()

- 은행에서 메인 타겟으로 두는 연령대가 있었고 20~50대가 그 범위였다. 
- 그 중에서도 특히 연락을 많이 돌린 연령대가 3040이다. 
- 연락을 많이 돌린다면, 그만큼 가입하는 사람도 많겠지? 

## 세분류

### 20~50대와 다른 연령대의 연락 비율 차이

In [ ]:
# 'Target'과 'Others'로 이진 분류
main_age_group = [20, 30, 40, 50]
bank_df['is_target'] = bank_df['age_group'].apply(lambda x: '20~50대 합산' if x in main_age_group else 'Others')

# 비율 계산
target_counts = bank_df['is_target'].value_counts()

# 시각화 (파이차트)
plt.figure(figsize=(8, 8))
plt.pie(target_counts, labels=target_counts.index, autopct='%1.1f%%', startangle=90, 
        explode=[0.1, 0], shadow=True,)

plt.title('Total Campaign Reach: Main Target vs Others', fontsize = 16, y = 1.03)
plt.show()

### 메인 타겟들 중에서도 특히 연락이 많은 비율

In [ ]:
# 세 그룹으로 분류하는 함수 정의
def age_classifier(age):
    if age in [30, 40]:
        return '3~40대'
    elif age in [20, 50]:
        return '20대 or 50대'
    else:
        return 'Others'

bank_df['target_segment'] = bank_df['age_group'].apply(age_classifier)

# 비율 계산 (내림차순 정렬)
segment_counts = bank_df['target_segment'].value_counts()

# 시각화 (파이차트)
plt.figure(figsize=(9, 9))
plt.pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%', startangle=140, explode=[0.05, 0.05, 0.05], 
        shadow=True,textprops={'fontsize': 12, 'weight': 'bold'})

plt.title('Campaign Reach by Segment: 3040 vs 2050 vs Others', fontsize=16, y = 1.03)
plt.show()

### countplot

In [ ]:
# 연령대로 그룹바이
bank_df_age = bank_df['age_group'].value_counts()

# countplot
plt.figure(figsize=(12, 8))
ax = sns.countplot(bank_df, x = 'age_group', hue = 'age_group', legend=False, palette = "Blues")
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', fontsize=9, color='black', xytext=(0, 5), 
                textcoords='offset points')
plt.title('Campaign distribution by Age', fontsize = 16, y = 1.03)
plt.xlabel('Age group', fontsize=12)
plt.ylabel('Campaign count', fontsize=12)
sns.despine()

# 비율이 많은 연령대(2~50)대와 그렇지 않은 연령대의 비율 차이 확인 
main_age_group = [20, 30, 40, 50] # 2, 3, 4, 50대
bank_df['group_category'] = bank_df['age_group'].apply(lambda x: f"{int(x)}대" if x in main_age_group else '기타(Others)') 
# 저 안에 있는 람다는 익명함수, 혹은 람다식이라고 부릅니다. 저는 김람다씨라고 부르고 있죠. 
# 기명함수 써도 되는데 왜 람다냐... 한줄로 끝나는거라서요. 
order_list = ['20대', '30대', '40대', '50대', '기타(Others)']

# 20~50대+기타로 묶음
group_counts = bank_df['group_category'].value_counts()
plt.figure(figsize=(12, 6))

# 시각화
ax = sns.countplot(data=bank_df, x='group_category', hue='group_category', hue_order=order_list,
                    order=order_list, legend=False, palette=['#9dadbc', '#5ca0de','#5ca0de', '#5ca0de', '#9dadbc'])

# 그래프 막대기마다 값 추가하는 코드(주석처리됨)
# for p in ax.patches:
#     ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
#                 ha='center', va='center', fontsize=9, color='black', xytext=(0, 5), 
#                 textcoords='offset points')
# plt.title('연령대별 연락 횟수 분포', fontsize = 16, y = 1.03)
plt.xlabel('')
plt.ylabel('')
sns.despine()

plt.show()

# 연령대별 가입률

In [ ]:
# 연령대와 성공률로 묶음
age_group_deposit = bank_df.groupby(['group_category', 'deposit']).size().unstack()

# 성공률 계산
age_group_deposit['sum'] = age_group_deposit['no'] + age_group_deposit['yes']
age_group_deposit['yes_rate'] = round((age_group_deposit['yes'] / age_group_deposit['sum']) * 100, 3)

# 성공률 순으로 정렬 (내림차순)
age_group_deposit_s = age_group_deposit.sort_values('yes_rate', ascending=False)
age_group_deposit_s

In [ ]:
plt.figure(figsize=(12, 6))

ax = sns.barplot(data= age_group_deposit_s, x='group_category', y='yes_rate',legend=False, hue = 'group_category', hue_order=order_list,
                order=order_list, palette=['#9dadbc', '#5ca0de','#5ca0de', '#5ca0de', '#9dadbc'])

# 각 막대에 수치 추가
# for container in ax.containers:
#     ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=10, fontweight='bold')

plt.xlabel('')
plt.ylabel('')
# plt.title('고객 연령대별 예금 가입률', fontsize=16, y = 1.03)
plt.ylim(0, 85)
sns.despine()

plt.show()

### 그래프 병합

In [ ]:
# 인덱스로 빠져있을 수 있는 group_category를 컬럼으로 복구
if 'group_category' not in age_group_deposit_s.columns:
    age_group_deposit_s = age_group_deposit_s.reset_index()

# 컨택 횟수(빈도) 데이터 생성 및 병합
count_data = bank_df['group_category'].value_counts().reset_index()
count_data.columns = ['group_category', 'contact_count']

# 두 데이터프레임을 하나로 병합 (연령대 기준)
final_df = age_group_deposit_s.merge(count_data, on='group_category')

# order_list 순서에 맞게 정렬
final_df['group_category'] = pd.Categorical(final_df['group_category'], categories=order_list, ordered=True)
final_df = final_df.sort_values('group_category')

# 막대그래프(연락 빈도)
fig, ax1 = plt.subplots(figsize=(12, 6))

# 막대 그래프: 컨택 횟수 (왼쪽 Y축)
sns.barplot(data=final_df, x='group_category', y='contact_count', 
            alpha=1.0, ax=ax1, hue='group_category', legend=False, palette=['#9dadbc', '#5ca0de','#5ca0de', '#9dadbc', '#9dadbc'])
ax1.set_ylabel('')
ax1.set_xlabel('')

# 막대 수치 표시
for p in ax1.patches:
    ax1.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='bottom', fontsize=10, xytext=(0, 5), textcoords='offset points')

# 꺾은선 그래프: 가입률 (오른쪽 Y축)
ax2 = ax1.twinx()
sns.lineplot(data=final_df, x='group_category', y='yes_rate', 
                color='#ee0000', marker='o', markersize=8, linewidth=2, ax=ax2)
ax2.set_ylabel('')
ax1.yaxis.grid(True, linestyle='--', alpha=0.5) 
ax2.yaxis.grid(False) # 오른쪽 축 그리드는 제거 (겹침 방지)

# 가입률 수치 표시
for i, rate in enumerate(final_df['yes_rate']):
    ax2.text(i, rate + 1, f'{rate:.1f}%', color='red', ha='center', fontweight='bold')

# plt.title('Campaign Contacts vs. Deposit Success Rate', fontsize=16, pad=20)
sns.despine()

plt.show()

- 특정 연령대(3~40대)인 고객들의 가입률이 높을 것이다->특정 연령대의 고객 가입률이 높은 건 맞는데, 은행에서 주요 타겟으로 하는 연령대는 아니었다. 
- 10대+60대 이상이 가장 높았고, 그 다음으로 20대, 30대, 50대, 40대 순으로 제일 낮았다. 
- 은행의 주 타겟 연령대 중에서는 20대만 5할 이상이고, 나머지는 4할 언저리다. 20대도 연락 횟수를 생각하면 타율이 그렇게 높은 편은 아니다. 
> 결혼, 노후자금 등의 요인으로 제일 가입을 많이 할 거라고 생각했던 3~40대의 가입률이 오히려 저조했다. 

# 3040을 끌어들일 전략

- 그럼 이들을 고객으로 끌어들이려면 어떤 전략을 시도해야 할까? 

## 메인 연령대의 직업별 deposit 비율

In [ ]:
# 메인 타겟의 직업/deposit 비율

# 20~50대만 필터링
main_target = bank_df[bank_df['age_group'].isin([20, 30, 40, 50])]

# 시각화
ax = sns.countplot(data=main_target, x='job', hue='deposit')

# 각 작대기에 값 추가 
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', fontsize=9, color='black', xytext=(0, 5), 
                textcoords='offset points')

plt.xticks(rotation=45)
plt.title('Job Distribution by Deposit Success (Age 20-50)', fontsize=16, y = 1.03)
plt.xlabel('Job', fontsize = 12)
plt.ylabel('Deposit count', fontsize = 12)
sns.despine()
plt.show()

- 직업별로 확인해보니, 학생, 취준생(unemployeed, 편의상 취준생으로 번역)을 제외한 모든 직군에서 가입하지 않은 고객들의 비용이 높았다. 
- admin, technician, services, managemant, blue-collar의 막대기가 특히 높았다. 트라이한 직업이 많다는 얘기.. 
- blue-collar가 yes, no의 격차가 제일 컸다. 
- 그러니까 우리는 저 분들의 deposit을 yes로 만들어 줄 확실한 전략이 필요하다, 이거다. 

### 직업별 가입비
- 유튜브에는 좋/싫비라는 게 있다. 그냥 좋아요 대비 싫어요의 비율이죠, 뭐. 
- 그런것처럼 직업별로 yes/no의 비율을 구할거다. 당연하게도 no가 많으면 비율이 1 아래로 내려간다. 

In [ ]:
# 묶어봅니다
main_target_cnt = main_target.groupby(['job', 'deposit']).size().unstack(fill_value=0)

# 2. 좋싫비... 아니고 가입비. (유튜브에서는 좋싫비)
main_target_cnt['Y/N_rate'] = main_target_cnt['yes'] / main_target_cnt['no']
main_target_final = main_target_cnt.sort_values(by='Y/N_rate', ascending=False)

main_target_final

In [ ]:
# 시각화
ax = sns.barplot(data= main_target_final, x='job', y='Y/N_rate',legend=False, hue = 'job')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.xticks(rotation=45)
plt.xlabel('Job', fontsize = 12)
plt.ylabel('Yes/No rate', fontsize = 12)
plt.title('Deposit Success Rate by Job', fontsize=16, y = 1.03)
sns.despine()
plt.show()

- 학생이 다른 직업군 대비 가입비가 압도적으로 높다. unemployeed랑 비교해도 거의 2배 넘겼다. 

## 메인 연령대의 통화 시간 대비 deposit
- (20~50대 중) 연령대 상관 없이 통화 시간에 따라 가입비가 어떤지만 볼 예정입니다. 

In [ ]:
# 통화 시간 범주화
main_target = bank_df[bank_df['age_group'].isin([20, 30, 40, 50])].copy()

bins = list(np.linspace(2, 3881, 11))
labels = [f"{int(bins[i]//60)}~{int(bins[i+1]//60)}분" for i in range(len(bins)-1)]
main_target['duration_level'] = pd.cut(bank_df['duration'], bins = bins, labels = labels, right=False)

# groupby
main_target_cnt = main_target.groupby(['duration_level', 'deposit'], observed=False).size().unstack(fill_value=0)

# 시각화 
ax = sns.countplot(data=main_target, x='duration_level', hue='deposit')

# 기존에 잘 나오던 annotate 방식에 if 조건문만 추가 (얘만 안돼서요...)
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 0인 값(데이터 없는 구간)은 무시
        ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height), ha='center', va='center', fontsize=9, color='black', 
                    fontweight='bold', xytext=(0, 7), textcoords='offset points')

plt.xticks(rotation=45)
plt.title('Deposit Success by Duration (Age 20-50)', fontsize=16, y = 1.03)
plt.xlabel('Job', fontsize = 12)
plt.ylabel('Deposit count', fontsize = 12)
sns.despine()
plt.show()

In [ ]:
# 그래프 y축을 로그로 조정(왼쪽이 너무 커서 오른쪽이 안보여요)
ax = sns.countplot(data=main_target, x='duration_level', hue='deposit')
ax.set_yscale("log") 

# 기존에 잘 나오던 annotate 방식에 if 조건문만 추가
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 0인 값(데이터 없는 구간)은 무시
        ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height), ha='center', va='center', fontsize=9, color='black', 
                    fontweight='bold', xytext=(0, 7), textcoords='offset points')

plt.xticks(rotation=45)
plt.title('Deposit Success by Duration (Age 20-50, Log Scale)', fontsize=16, y = 1.03)
plt.xlabel('Duration level', fontsize = 12)
plt.ylabel('Deposit count(log)', fontsize = 12)
sns.despine()
plt.show()

- 통화 시간이 0~6분 이내일 때가 값이 제일 높았다. 그 뒤로는 점점 떨어지고 있지만, 6~12분부터는 가입비가 늘어나고 있다. 
- 관심 없는 고객이라는 단칼에 끊을테니 0~6분 사이에서는 싫어요가 높고, 관심 있는 고객이라면 이것저것 따져보느라 통화 시간이 길어져서 그런 게 아닐까? 

### 통화시간 대비 가입비 변화

In [ ]:
# 통화 시간대별 가입비
# 38분 이후를 하나의 범주로 합치기
main_target_dur = main_target.copy()
main_target_dur['duration_level_refined'] = main_target_dur['duration'].apply(
    lambda x: '38분 이상' if x >= 38*60 else pd.cut([x], bins=bins, labels=labels, right=False)[0]
)

target_stats = main_target_dur.groupby(['duration_level_refined', 'deposit'], observed=False).size().unstack(fill_value=0)

# 가입비 도출
target_stats['Y/N_rate'] = target_stats['yes'] / target_stats['no']

correct_order = labels[:6] + ['38분 이상']
target_stats = target_stats.reset_index()
target_stats['duration_level_refined'] = pd.Categorical(
    target_stats['duration_level_refined'], 
    categories=correct_order, 
    ordered=True
)
target_stats = target_stats.sort_values('duration_level_refined')

# 시각화 
ax = sns.lineplot(target_stats, x = 'duration_level_refined', y = 'Y/N_rate', marker = 'o')

plt.xticks(rotation=45)
plt.title('Duration level by Deposit Success (Age 20-50)', fontsize=16, y = 1.03)
plt.xlabel('Durarion(min)', fontsize = 12)
plt.ylabel('Deposit rate', fontsize = 12)
plt.grid(True, linestyle='--', alpha=0.5)
sns.despine()
plt.show()

- 개인적으로 통화 시간은 좋싫비에 영향을 끼치는 게 아니라, 좋싫비에 영향을 받는? 요소라고 생각합니다. 
- 관심 없는 전화였다면 여보세요 은행입니'다'에서 끊었을테고, 가입을 고려중이거나 다른 은행이랑 비교중이라면 이것저것 재기 위해 물어볼 게 많아질테니 통화시간도 길어지겠죠. 
> 단, 그렇다고 해서 관심 없는 고객 붙들고 질질 끌라는 얘기는 아님. (고객의 피로도만 증가함)

### 전 연령대로 확장

In [ ]:
# 통화 시간 범주화
duration_all = bank_df.copy()

bins = list(np.linspace(2, 3881, 11))
labels = [f"{int(bins[i]//60)}~{int(bins[i+1]//60)}분" for i in range(len(bins)-1)]
duration_all['duration_level'] = pd.cut(bank_df['duration'], bins = bins, labels = labels, right=False)

# 묶어주고
main_target_cnt_all = duration_all.groupby(['duration_level', 'deposit'], observed=False).size().unstack(fill_value=0)

# 시각화
ax = sns.countplot(data=duration_all, x='duration_level', hue='deposit')

# 위에 그거... (막대에 값 써주는...)
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 0인 값(데이터 없는 구간)은 무시
        ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height), ha='center', va='center', fontsize=9, color='black', 
                    fontweight='bold', xytext=(0, 7), textcoords='offset points')

plt.xticks(rotation=45)
plt.title('Job Distribution by Deposit Success (Age 20-50)', fontsize=16, y = 1.03)
plt.xlabel('Job', fontsize = 12)
plt.ylabel('Deposit count', fontsize = 12)
sns.despine()
plt.show()

In [ ]:
# 통화 시간대별 좋싫비
# 38분 이후를 하나의 범주로 합치기
main_target_dur = duration_all.copy()
main_target_dur['duration_level_refined'] = main_target_dur['duration'].apply(
    lambda x: '38분 이상' if x >= 38*60 else pd.cut([x], bins=bins, labels=labels, right=False)[0]
)

target_stats = main_target_dur.groupby(['duration_level_refined', 'deposit'], observed=False).size().unstack(fill_value=0)

# 가입비
target_stats['Y/N_rate'] = target_stats['yes'] / target_stats['no']

# 시간 순서대로 정렬(이거 안 하면 순서 이상해요)
correct_order = labels[:6] + ['38분 이상']
target_stats = target_stats.reset_index()
target_stats['duration_level_refined'] = pd.Categorical(
    target_stats['duration_level_refined'], 
    categories=correct_order, 
    ordered=True
)
target_stats = target_stats.sort_values('duration_level_refined')

# 시각화
ax = sns.lineplot(target_stats, x = 'duration_level_refined', y = 'Y/N_rate', marker = 'o')

plt.xticks(rotation=45)
plt.title('Duration level by Deposit Success (Age 20-50)', fontsize=16, y = 1.03)
plt.xlabel('Durarion(min)', fontsize = 12)
plt.ylabel('Deposit rate', fontsize = 12)
plt.grid(True, linestyle='--', alpha=0.5)
sns.despine()
plt.show()

- 연령대를 확장해봐도 결과는 똑같았고, 오히려 0~6분대 가입비가 주 타겟 연령층만 봤을때에 비해 좀 더 작아졌다. (싫어요가 더 많았음)
- 꺾은선그래프 역시 32~38분에서 잠깐 하락세를 보였던 걸 제외하면 양상은 비슷했다. 
> 마의 6분입니다. 

## 과거 마케팅 이력 대비 가입비

### 과거 마케팅 이력 대비 가입자 수

In [ ]:
# 통화 시간 범주화
main_target = bank_df[bank_df['age_group'].isin([20, 30, 40, 50])].copy()

# groupby
main_target_outcome = main_target.groupby(['poutcome', 'deposit'], observed=False).size().unstack(fill_value=0)

# coutplot으로 시각화
ax = sns.countplot(data=main_target, x='poutcome', hue='deposit')
ax.set_yscale("log") 

# 기존에 잘 나오던 annotate 방식에 if 조건문만 추가
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 0인 값(데이터 없는 구간)은 무시
        ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height), ha='center', va='center', fontsize=9, color='black', 
                    fontweight='bold', xytext=(0, 7), textcoords='offset points')

plt.title('Past outcome by Deposit success (Age 20-50)', fontsize=16, y = 1.03)
plt.xlabel('poutcome', fontsize = 12)
plt.ylabel('Deposit count', fontsize = 12)
sns.despine()
plt.show()

### 과거 마케팅 이력 대비 가입비

In [ ]:
# 통화 시간 범주화
main_target = bank_df[bank_df['age_group'].isin([20, 30, 40, 50])].copy()

# groupby
main_target_outcome = main_target.groupby(['poutcome', 'deposit'], observed=False).size().unstack(fill_value=0)

main_target_outcome['Y/N_rate'] = main_target_outcome['yes'] / main_target_outcome['no']
main_target_final_o = main_target_outcome.sort_values(by='Y/N_rate', ascending=False)

# coutplot
ax = sns.barplot(data=main_target_final_o, x='poutcome',y='Y/N_rate', hue='poutcome')

# 기존에 잘 나오던 annotate 방식에 if 조건문만 추가
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 0인 값(데이터 없는 구간)은 무시
        ax.annotate(f'{round(float(height),3)}', (p.get_x() + p.get_width() / 2., height), ha='center', va='center', fontsize=9, color='black', 
                    fontweight='bold', xytext=(0, 7), textcoords='offset points')

plt.title('Past outcome by Deposit success (Age 20-50)', fontsize=16, y = 1.03)
plt.xlabel('poutcome', fontsize = 12)
plt.ylabel('Deposit count', fontsize = 12)
sns.despine()
plt.show()

- other, success를 제외하면 다 yes 대비 no가 높다. 
- 높은 순서는 success > other > failure > unknown. 

### unknown 그룹의 통화 시간 대비 가입비 변화

In [ ]:
# poutcome이 unknown인 사람들 중, 통화 시간이 길어짐에 따라 가입률이 어떻게 변하는지 확인
unknown_group = main_target[main_target['poutcome'] == 'unknown'].copy()

# 통화 시간대별 좋싫비
# 38분 이후를 하나의 범주로 합치기
unknown_group['duration_level_refined'] = unknown_group['duration'].apply(
    lambda x: '38분 이상' if x >= 38*60 else pd.cut([x], bins=bins, labels=labels, right=False)[0]
)

target_stats = unknown_group.groupby(['duration_level_refined', 'deposit'], observed=False).size().unstack(fill_value=0)

# 가입비
target_stats['Y/N_rate'] = target_stats['yes'] / target_stats['no']

correct_order = labels[:6] + ['38분 이상']
target_stats = target_stats.reset_index()
target_stats['duration_level_refined'] = pd.Categorical(target_stats['duration_level_refined'], categories=correct_order, ordered=True)
target_stats = target_stats.sort_values('duration_level_refined')

# 그래~프~ 
ax = sns.lineplot(target_stats, x = 'duration_level_refined', y = 'Y/N_rate', marker = 'o')

plt.xticks(rotation=45)
plt.title('Duration time by Deposit Success (Age 20-50): poutcome unknown', fontsize=16, y = 1.03)
plt.xlabel('Durarion(min)', fontsize = 12)
plt.ylabel('Deposit rate', fontsize = 12)
plt.grid(True, linestyle='--', alpha=0.5)
sns.despine()
plt.show()

- 언노운만 떼어놓고 봤을 때의 양상도 비슷하다. 

### duration time(세분류)

In [ ]:
# 통화 시간 범주화
main_target = bank_df[bank_df['age_group'].isin([20, 30, 40, 50])].copy()

bins = list(np.linspace(2, 3881, 31))
labels = [f"{int(bins[i]//60)}~{int(bins[i+1]//60)}분" for i in range(len(bins)-1)]
main_target['duration_level'] = pd.cut(bank_df['duration'], bins = bins, labels = labels, right=False)

# groupby
main_target_cnt = main_target.groupby(['duration_level', 'deposit'], observed=False).size().unstack(fill_value=0)

# 그래프
plt.figure(figsize=(18, 11))
ax = sns.countplot(data=main_target, x='duration_level', hue='deposit')
ax.set_yscale("log") 

# 기존에 잘 나오던 annotate 방식에 if 조건문만 추가
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 0인 값(데이터 없는 구간)은 무시
        ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height), ha='center', va='center', fontsize=9, color='black', fontweight='bold', 
                    xytext=(0, 7), textcoords='offset points')

plt.xticks(rotation=45)
plt.title('Deposit Rate by Call Duration (Age 20-50)', fontsize=16, y = 1.03)
plt.xlabel('Job', fontsize = 12)
plt.ylabel('Deposit count', fontsize = 12)
sns.despine()
plt.show()

In [ ]:
# 전 연령대
main_target_dur = duration_all.copy()

bins = list(np.linspace(2, 3881, 31))
labels = [f"{int(bins[i]//60)}~{int(bins[i+1]//60)}분" for i in range(len(bins)-1)]
main_target_dur['duration_level'] = pd.cut(bank_df['duration'], bins = bins, labels = labels, right=False)

# groupby
main_target_cnt = main_target_dur.groupby(['duration_level', 'deposit'], observed=False).size().unstack(fill_value=0)

# 시각화
plt.figure(figsize=(18, 11))
ax = sns.countplot(data=main_target_dur, x='duration_level', hue='deposit')
ax.set_yscale("log") 

# 기존에 잘 나오던 annotate 방식에 if 조건문만 추가
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 0인 값(데이터 없는 구간)은 무시
        ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height), ha='center', va='center', fontsize=9, color='black', 
                    fontweight='bold', xytext=(0, 7), textcoords='offset points')

plt.xticks(rotation=45)
plt.title('Conversion Rate by Call Duration', fontsize=16, y = 1.03)
plt.xlabel('Job', fontsize = 12)
plt.ylabel('Deposit count', fontsize = 12)
sns.despine()
plt.show()

In [ ]:
# 통화 시간대별 좋싫비
# 38분 이후를 하나의 범주로 합치기
main_target_dur = bank_df.copy()
main_target_dur['duration_level_refined'] = main_target_dur['duration'].apply(
    lambda x: '38분 이상' if x >= 38*60 else pd.cut([x], bins=bins, labels=labels, right=False)[0]
)

target_stats = main_target_dur.groupby(['duration_level_refined', 'deposit'], observed=False).size().unstack(fill_value=0)

# 가입비
target_stats['Y/N_rate'] = target_stats['yes'] / target_stats['no']

# 얘도 이거 안 해주면 순서 꼬여유... 
correct_order = labels[:18] + ['38분 이상']
target_stats = target_stats.reset_index()
target_stats['duration_level_refined'] = pd.Categorical(
    target_stats['duration_level_refined'], 
    categories=correct_order, 
    ordered=True
)
target_stats = target_stats.sort_values('duration_level_refined')

# 시각화
plt.figure(figsize=(18, 11))
ax = sns.lineplot(target_stats, x = 'duration_level_refined', y = 'Y/N_rate', marker = 'o')

plt.xticks(rotation=45)
plt.title('Conversion Rate by Call Duration', fontsize=16, y = 1.03)
plt.xlabel('Durarion(min)', fontsize = 12)
plt.ylabel('Deposit rate', fontsize = 12)
plt.grid(True, linestyle='--', alpha=0.5)
sns.despine
plt.show()

In [ ]:
# 통화 시간대별 좋싫비
# 38분 이후를 하나의 범주로 합치기
main_target_dur = main_target.copy()
main_target_dur['duration_level_refined'] = main_target_dur['duration'].apply(
    lambda x: '38분 이상' if x >= 38*60 else pd.cut([x], bins=bins, labels=labels, right=False)[0]
)

target_stats = main_target_dur.groupby(['duration_level_refined', 'deposit'], observed=False).size().unstack(fill_value=0)

# 가입비
target_stats['Y/N_rate'] = target_stats['yes'] / target_stats['no']

correct_order = labels[:18] + ['38분 이상']
target_stats = target_stats.reset_index()
target_stats['duration_level_refined'] = pd.Categorical(
    target_stats['duration_level_refined'], 
    categories=correct_order, 
    ordered=True
)
target_stats = target_stats.sort_values('duration_level_refined')

# 시각화
plt.figure(figsize=(18, 11))
ax = sns.lineplot(target_stats, x = 'duration_level_refined', y = 'Y/N_rate', marker = 'o')

plt.xticks(rotation=45)
plt.title('Conversion Rate by Call Duration (Age 20-50)', fontsize=16, y = 1.03)
plt.xlabel('Durarion(min)', fontsize = 12)
plt.ylabel('Deposit rate', fontsize = 12)
plt.grid(True, linestyle='--', alpha=0.5)
sns.despine()
plt.show()

## contact 방법 대비 성공률

In [ ]:
# 전체 접근 방법
main_target_rate = main_target.groupby(['contact', 'deposit'])['balance'].size().unstack(fill_value=0)

main_target_rate['total'] = main_target_rate['yes'] + main_target_rate['no']
main_target_rate['Y/N_rate'] = (main_target_rate['yes'] / main_target_rate['total']) * 100
main_target_final_r = main_target_rate.sort_values(by='Y/N_rate', ascending=False)

# 그래프
ax = sns.barplot(main_target_final_r, x = 'contact', y = 'Y/N_rate', hue = 'contact')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by contact', fontsize=16, y=1.03)
plt.ylabel('Deposit Percentage (%)')
sns.despine()
plt.show()

### 세분류(contact-연령)

In [ ]:
# 접촉 방법+연령
main_target_rate = main_target.groupby(['group_category', 'contact', 'deposit'])['balance'].size().unstack(fill_value=0)

main_target_rate['total'] = main_target_rate['yes'] + main_target_rate['no']
main_target_rate['Y/N_rate'] = (main_target_rate['yes'] / main_target_rate['total']) * 100
main_target_final_r = main_target_rate.sort_values(by='Y/N_rate', ascending=False)

# 그래프으으
ax = sns.barplot(main_target_final_r, x = 'contact', y = 'Y/N_rate', hue = 'group_category')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by contact', fontsize=16, y=1.03)
plt.ylabel('Deposit percentage (%)')
plt.ylim(0, 75)
sns.despine()
plt.show()

### 세분류(연령-contact)

In [ ]:
# 연령대+접촉방법
main_target_rate = main_target.groupby(['group_category', 'contact', 'deposit'])['balance'].size().unstack(fill_value=0)

main_target_rate['total'] = main_target_rate['yes'] + main_target_rate['no']
main_target_rate['Y/N_rate'] = (main_target_rate['yes'] / main_target_rate['total']) * 100
main_target_final_r = main_target_rate.sort_values(by='Y/N_rate', ascending=False)

# 그래프
ax = sns.barplot(main_target_final_r, x = 'group_category', y = 'Y/N_rate', hue = 'contact')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by contact', fontsize=16, y=1.03)
plt.ylabel('Deposit percentage (%)')
plt.ylim(0, 75)
sns.despine()
plt.show()

- cellular-telephone-unknown순으로 성공률이 높았으며, 연령대 상관없이 비슷한 양상을 보였다. 

## 결혼여부에 따른 성공률

In [ ]:
# 결혼여부
main_target_rate = main_target.groupby(['marital', 'deposit'])['balance'].size().unstack(fill_value=0)

main_target_rate['total'] = main_target_rate['yes'] + main_target_rate['no']
main_target_rate['Y/N_rate'] = main_target_rate['yes'] / main_target_rate['total'] * 100
main_target_final_r = main_target_rate.sort_values(by='Y/N_rate', ascending=False)

# 그래프
ax = sns.barplot(main_target_final_r, x = 'marital', y = 'Y/N_rate', hue = 'marital')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by Marital', fontsize=16, y=1.03)
plt.xlabel('Marital')
plt.ylabel('Deposit percentage (%)')
plt.ylim(0, 55)
sns.despine()
plt.show()

### 연령대별 세분류(연령-결혼)

In [ ]:
# 연령대+결혼유무
main_target_rate = main_target.groupby(['group_category', 'marital', 'deposit'])['balance'].size().unstack(fill_value=0)

main_target_rate['total'] = main_target_rate['yes'] + main_target_rate['no']
main_target_rate['Y/N_rate'] = main_target_rate['yes'] / main_target_rate['total'] * 100

# 그래프
ax = sns.barplot(main_target_rate, x = 'group_category', y = 'Y/N_rate', hue = 'marital')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by Marital', fontsize=16, y=1.03)
plt.ylabel('Deposit percentage (%)')
sns.despine()
plt.show()

### 연령대별 세분류(결혼-연령)

In [ ]:
# 결혼+연령대
main_target_rate = main_target.groupby(['group_category', 'marital', 'deposit'])['balance'].size().unstack(fill_value=0)

main_target_rate['total'] = main_target_rate['yes'] + main_target_rate['no']
main_target_rate['Y/N_rate'] = main_target_rate['yes'] / main_target_rate['no']

# 그래프
ax = sns.barplot(main_target_rate, x = 'marital', y = 'Y/N_rate', hue = 'group_category')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by Marital', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
sns.despine()
plt.show()

- 50대를 제외하고 결혼 여부에 따른 가입비 변화 양상은 싱글 > 이혼 > 기혼순이었다. 
- 20대여도 이혼했거나 결혼한 사람들은 싱글 대비 가입비가 훅 떨어졌다. 

## 기타 다른 요인들에 따른 가입비

### 학력

In [ ]:
# 전체 접근 방법
main_target_edu = main_target.groupby(['education', 'deposit'])['balance'].size().unstack(fill_value=0)

main_target_edu['total'] = main_target_edu['yes'] + main_target_edu['no']
main_target_edu['Y/N_rate'] = main_target_edu['yes'] / main_target_edu['no']
main_target_final_r = main_target_edu.sort_values(by='Y/N_rate', ascending=False)

# 그래프
ax = sns.barplot(main_target_final_r, x = 'education', y = 'Y/N_rate', hue = 'education')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by Education', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
sns.despine()
plt.show()

### 세분류-나이

In [ ]:
# 연령대+학력
main_target_edu = main_target.groupby(['group_category','education', 'deposit'])['balance'].size().unstack(fill_value=0)

main_target_edu['total'] = main_target_edu['yes'] + main_target_edu['no']
main_target_edu['Y/N_rate'] = main_target_edu['yes'] / main_target_edu['no']
main_target_final_r = main_target_edu.sort_values(by='Y/N_rate', ascending=False)

# 그래프
ax = sns.barplot(main_target_final_r, x = 'group_category', y = 'Y/N_rate', hue = 'education')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by Education', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
sns.despine()
plt.show()

- tertiary가 뭔 의미임? 3차 교육과정 이런건 아닐테고... 
- 학력 수준과 가입비는 정비례 관계인데, 문제는 저 언노운이 누구인가다. 
- 3~40대의 경우 tetiary는 가입비가 1을 넘겼다. 

### default(신용불량)

In [ ]:
# 신용불량 그룹바이 
main_target_def = main_target.groupby(['default', 'deposit'])['balance'].size().unstack(fill_value=0)

main_target_def['total'] = main_target_def['yes'] + main_target_def['no']
main_target_def['Y/N_rate'] = main_target_def['yes'] / main_target_def['no']
main_target_final_r = main_target_def.sort_values(by='Y/N_rate', ascending=False)

# 하고 그래프
ax = sns.barplot(main_target_final_r, x = 'default', y = 'Y/N_rate', hue = 'default')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by Default', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
plt.ylim(0, 0.85)
sns.despine()
plt.show()

### 세분류

In [ ]:
# 연령대+신불자 여부
main_target_def = main_target.groupby(['group_category', 'default', 'deposit'])['balance'].size().unstack(fill_value=0)

main_target_def['total'] = main_target_def['yes'] + main_target_def['no']
main_target_def['Y/N_rate'] = main_target_def['yes'] / main_target_def['no']
main_target_final_r = main_target_def.sort_values(by='Y/N_rate', ascending=False)

# 그래프
ax = sns.barplot(main_target_final_r, x = 'group_category', y = 'Y/N_rate', hue = 'default')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by Default', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
sns.despine()
plt.ylim(0, 1.55)
plt.show()

- 전체적으로 신불자가 아닌(no) 사람이 신불자(yes)에 비해 가입비가 2배가량 높았다. 
- 갭은 40대가 제일 컸고, 전체적인 가입비는 20대가 더 컸다. 
- 20대가 왜 신불자가 있음? 포르투갈에도 카푸어 그런 게 있나? 

## campaign(접촉횟수)

In [ ]:
# 접촉 횟수
main_target_cam = main_target.copy()
main_target_cam['campaign_category'] = bank_df['campaign'].apply(lambda x: f"{int(x)}회" if x < 3 else '3회 이상') 
main_target_cam = main_target_cam.groupby(['campaign_category', 'deposit'])['balance'].size().unstack(fill_value=0)

# 가입비
main_target_cam['total'] = main_target_cam['yes'] + main_target_cam['no']
main_target_cam['Y/N_rate'] = main_target_cam['yes'] / main_target_cam['no']
main_target_final_r = main_target_cam.sort_values(by='Y/N_rate', ascending=False)

# 그래프으으 
ax = sns.barplot(main_target_final_r, x = 'campaign_category', y = 'Y/N_rate', hue = 'campaign_category')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by campaign', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
sns.despine()
plt.show()

In [ ]:
# 연령대+접촉 횟수 
main_target_cam = main_target.copy()
main_target_cam['campaign_category'] = bank_df['campaign'].apply(lambda x: f"{int(x)}회" if x < 3 else '3회 이상') 
main_target_cam = main_target_cam.groupby(['group_category','campaign_category', 'deposit'])['balance'].size().unstack(fill_value=0)

# 가입비
main_target_cam['total'] = main_target_cam['yes'] + main_target_cam['no']
main_target_cam['Y/N_rate'] = main_target_cam['yes'] / main_target_cam['no']

# 그래프
ax = sns.barplot(main_target_cam, x = 'group_category', y = 'Y/N_rate', hue = 'campaign_category')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by campaign', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
sns.despine()
plt.show()

- 접촉 횟수와 가입비는 확실히 반비례한다. 20대의 경우 2회~3회 이상으로 가면서 낙차가 두드러지는 편. 

### previous group

In [ ]:
# 이전 접촉 여부
main_target_prev = main_target.copy()
main_target_prev = main_target_prev.groupby(['previous_group', 'deposit'])['balance'].size().unstack(fill_value=0)

# 가입비+정렬 
main_target_prev['total'] = main_target_prev['yes'] + main_target_prev['no']
main_target_prev['Y/N_rate'] = main_target_prev['yes'] / main_target_prev['no']
main_target_final_r = main_target_prev.sort_values(by='Y/N_rate', ascending=False)

# 그래프
ax = sns.barplot(main_target_final_r, x = 'previous_group', y = 'Y/N_rate', hue = 'previous_group')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by Previous group', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
sns.despine()
plt.show()

In [ ]:
# 연령대로 세분화 
main_target_prev = main_target.copy()
main_target_prev = main_target_prev.groupby(['group_category','previous_group', 'deposit'])['balance'].size().unstack(fill_value=0)

# 가입비 
main_target_prev['total'] = main_target_prev['yes'] + main_target_prev['no']
main_target_prev['Y/N_rate'] = main_target_prev['yes'] / main_target_prev['no']

# 그래프
ax = sns.barplot(main_target_prev, x = 'group_category', y = 'Y/N_rate', hue = 'previous_group')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by Previous group', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
sns.despine()
plt.show()

- 접촉 횟수는 클수록 가입률이 반비례하는데, 얘는 반대로 횟수를 늘릴수록 가입룰이 증가한다. 

### campaign+previous group

In [ ]:
# 접촉 횟수+이전 접촉 여부
main_target_prev = main_target.copy()
main_target_prev['campaign_category'] = bank_df['campaign'].apply(lambda x: f"{int(x)}회" if x < 3 else '3회 이상') 
main_target_prev = main_target_prev.groupby(['campaign_category','previous_group','deposit'])['balance'].size().unstack(fill_value=0)

# 가입비
main_target_prev['total'] = main_target_prev['yes'] + main_target_prev['no']
main_target_prev['Y/N_rate'] = main_target_prev['yes'] / main_target_prev['no']

# 그래프
ax = sns.barplot(main_target_prev, x = 'campaign_category', y = 'Y/N_rate', hue = 'previous_group')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by Previous group/Campaign', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
sns.despine()
plt.show()

- previous는 이번 회차에 접촉한 횟수, campaign은 이전에 접촉한 횟수이다. 
- previous의 경우 remarketing 그룹에서 오히려 가입률이 높았지만, campaign의 경우 횟수가 늘어날수록 오히려 가입률이 줄었다. 
- ~~연락 좀 작작 하라는 얘기~~ 모든것은 과유불급이라는 얘기. 

# 결론 (w/ Gemini)
## [통화 시간의 재발견] '마의 6분' 허들 존재

- 20~50대 직장인 타겟 분석 결과, 통화 시간 6분 미만 구간에서는 가입률이 극히 저조함.
- 6분 이후부터 가입률(Y/N Rate)이 기하급수적으로 상승하는 것으로 보아, 통화 지속 시간 자체가 고객의 '능동적 관심도'를 측정하는 핵심 지표임을 확인.

## [타겟팅 효율성] 무차별적 컨택의 비효율성 증명
- 과거 정보가 없는 'Unknown' 그룹과 'Success' 그룹 비교 시, 성공 이력이 있는 그룹의 효율이 약 10배 이상 높음.
- 정보가 없는 고객에게 무작위 전화를 돌리는 방식은 높은 피로도와 낮은 가성비를 초래하므로 지양해야 함.

## [전략적 제안] '선택과 집중'을 통한 리소스 최적화
- 모든 고객에게 전수 조사를 하기보다, 초반 3~5분 내에 반응이 없는 고객은 과감히 통화를 종료하여 마케팅 비용을 절감해야 함.
- 확보된 리소스를 6분 이상 대화를 지속하는 '고관여 고객'에게 집중 배치하여 최종 전환율을 극대화하는 전략 제안.

## [연락 수단] 고객에게 가장 효율적인 연락 수단은?
- 연령대를 막론하고 가장 효율적인 연락 수단은 cellular(핸드폰)이다. 
- 그것이 우리가 왜 스팸전화를 받는 이유이기도 하지... 

# 학생, 그들은 누구인가? 
- 20대~50대 중에서도 좋싫비가 압도적으로 높았던 직업군이 바로 학생이다. 그래서 학생과 다른 직군을 비교해 볼 예정이다. 

## 평균 통화시간

In [ ]:
# 학생과 학생이 아닌 직군으로 분리함
main_target_std = main_target.copy()
main_target_std['is_student'] = main_target_std['job'].apply(lambda x: 'Student' if x == 'student' else 'Others')

# 통화시간 평균
main_target_std.groupby('is_student')['duration'].mean('duration')

# 을 시각화
ax = sns.barplot(main_target_std, x = 'is_student', y = 'duration', hue = 'is_student')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')
    
plt.title('Average Call Duration: Student vs Others', fontsize=16, y=1.03)
plt.ylabel('Average Duration (sec)')
plt.show()

## 통화 시간 분포

### KDE

In [ ]:
plt.figure(figsize=(10, 6))

# x에 duration을 넣고 hue로 그룹을 나눕니다. fill=True로 색을 채우면 더 예뻐요.
sns.kdeplot(data=main_target_std, x='duration', hue='is_student', 
            fill=True, common_norm=False)

plt.title('Call Duration Distribution: Student vs Others', fontsize=16, y=1.03)
plt.xlabel('Duration (sec)')
plt.ylabel('Density')
plt.xlim(0, 2500) # 너무 긴 이상치는 잘라서 가독성을 높입니다.
plt.show()

# 학생이 뒤로 가 있는 이유: 사회인들 중에는 받자마자 끊는 사람도 많음 (추정)

### boxplot

In [ ]:
plt.figure(figsize=(10, 6))

# x에 카테고리, y에 수치형을 넣을 수 있습니다.
sns.boxplot(data=main_target_std, x='is_student', y='duration', 
                hue='is_student', width=0.3)

plt.title('Duration Density by Job Group (Box Plot)', fontsize=16, y=1.03)
plt.ylabel('Duration (sec)')
plt.ylim(0, 3000) # 이상치 때문에 그래프가 찌부되는 걸 방지
plt.show()

- 학생은 다른 직군에 비해 평균 통화시간이 약간 더 짧다. 
- 통화시간 분포를 boxplot으로 그렸을때도 상자의 높이가 다른 직군에 비해 더 낮았고, 이상치도 상대적으로 낮은 위치에 있다.
> 학생이 다른 직군에 비해 상대적으로 통화를 짧게 하는 편이다. 

## 평균 잔고

In [ ]:
# 학생과 학생이 아닌 직군으로 분리함
main_target_std = main_target.copy()
main_target_std['is_student'] = main_target_std['job'].apply(lambda x: 'Student' if x == 'student' else 'Others')

# 통화시간 평균
main_target_std.groupby('is_student')['balance'].mean('duration')

# 을 시각화
ax = sns.barplot(main_target_std, x = 'is_student', y = 'balance', hue = 'is_student')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')
    
plt.title('Average balance: Student vs Others', fontsize=16, y=1.03)
plt.ylabel('Average balance')
plt.show()

### Boxplot

In [ ]:
plt.figure(figsize=(10, 6))

# x에 카테고리, y에 수치형을 넣을 수 있습니다.
sns.boxplot(data=main_target_std, x='is_student', y='balance', 
                hue='is_student', width=0.3)

plt.title('Balance Density by Job Group (Box Plot)', fontsize=16, y=1.03)
plt.ylim(-10000, 20000) # 그래프 짜부 방지용
plt.ylabel('Balance')
plt.show()

- 학생이 다른 직군 대비 평균 잔고가 더 높았다. 
- boxplot을 확인해보니 다른 직군들은 마이너스 잔고가 있어서 그런거였다. 학생은 타 직군들과 달리 울타리도 0에 걸쳐있다. 
> 학생은 마통(마이너스 통장)이 없었다. 그것떄문에 평균 잔고가 높아보이는 것이었다. 

## 대출유무

In [ ]:
# 학생과 학생이 아닌 직군으로 분리함
main_target_std = main_target.copy()
main_target_std['is_student'] = main_target_std['job'].apply(lambda x: 'Student' if x == 'student' else 'Others')

# 대출여부 
ax = sns.countplot(main_target_std, x = 'is_student', hue = 'H/L')

# 기존에 잘 나오던 annotate 방식에 if 조건문만 추가
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 0인 값(데이터 없는 구간)은 무시
        ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height), ha='center', va='center', fontsize=9, color='black', 
                    fontweight='bold', xytext=(0, 7), textcoords='offset points')

plt.title('Loan by Job Group', fontsize=16, y=1.03)
plt.ylim(0, 5000)
plt.ylabel('H/L count')
plt.show()

### deposit=yes

In [ ]:
# 학생과 학생이 아닌 직군으로 분리함
main_target_std = main_target.copy()
main_target_std_yes = main_target_std.query('deposit == "yes"')
main_target_std_yes['is_student'] = main_target_std_yes['job'].apply(lambda x: 'Student' if x == 'student' else 'Others')

# 대출여부 
ax = sns.countplot(main_target_std_yes, x = 'is_student', hue = 'H/L')

# 기존에 잘 나오던 annotate 방식에 if 조건문만 추가
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 0인 값(데이터 없는 구간)은 무시
        ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height), ha='center', va='center', fontsize=9, color='black', 
                    fontweight='bold', xytext=(0, 7), textcoords='offset points')

plt.title('Loan by Job Group (Deposit=yes)', fontsize=16, y=1.03)
plt.ylim(0, 5000)
plt.ylabel('H/L count')
plt.show()

### deposit=no

In [ ]:
# 학생과 학생이 아닌 직군으로 분리함
main_target_std = main_target.copy()
main_target_std_no = main_target_std.query('deposit == "no"')
main_target_std_no['is_student'] = main_target_std_no['job'].apply(lambda x: 'Student' if x == 'student' else 'Others')

# 대출여부 
ax = sns.countplot(main_target_std_no, x = 'is_student', hue = 'H/L')

# 기존에 잘 나오던 annotate 방식에 if 조건문만 추가
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 0인 값(데이터 없는 구간)은 무시
        ax.annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height), ha='center', va='center', fontsize=9, color='black', 
                    fontweight='bold', xytext=(0, 7), textcoords='offset points')

plt.title('Loan by Job Group (Deposit=no)', fontsize=16, y=1.03)
plt.ylim(0, 5000)
plt.ylabel('H/L count')
plt.show()

- 학생은 다른 직군 대비 대출 수가 적다. 
- 대출이 있더라도 주담대+다른 대출을 병행하는 경우는 없다. (대출은 적어도 하나까지만)

## 접근 방법 대비 성공률

In [ ]:
# 전체 접근 방법
main_target_std = main_target.copy()
main_target_std['is_student'] = main_target_std['job'].apply(lambda x: 'Student' if x == 'student' else 'Others')
main_target_contact = main_target_std.groupby(['is_student','contact', 'deposit'])['balance'].size().unstack(fill_value=0)

# 가입비
main_target_contact['total'] = main_target_contact['yes'] + main_target_contact['no']
main_target_contact['Y/N_rate'] = (main_target_contact['yes'] / main_target_contact['total']) * 100
main_target_final_s = main_target_contact.sort_values(by='Y/N_rate', ascending=False)

# 그래프
ax = sns.barplot(main_target_final_s, x = 'is_student', y = 'Y/N_rate', hue = 'contact')

# 각 막대에 수치 추가
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10, fontweight='bold')

plt.title('success rate by contact', fontsize=16, y=1.03)
plt.ylabel('Deposit rate')
plt.show()

- 다른 직군에 비해 cellular, telephone의 성공률이 높다. 

# 더미데이터

- 딱히 쓸 예정은 없습니다. 

In [ ]:
# 과거 마케팅 결과(poutcome)에 따른 현재 통화 시간(duration)과 가입 여부 비교
plt.figure(figsize=(10, 6))
sns.boxplot(data=main_target, x='poutcome', y='duration', hue='deposit')
plt.ylim(0, 1500)
plt.title('Impact of Past Interest (poutcome) on Current Call Duration')
plt.show()